In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ─── Chemins ────────────────────────────────────────────────────────────────
BRUTES_PATH    = r"C:\Drone_Attack_Similarity_Project\DATASET\Brutes"
ATTACKS_PATH   = BRUTES_PATH + "\\Each_Attacks_CSV"
NORMAL_PATH    = BRUTES_PATH + "\\NomralAttacks"
UAVIDS_PATH    = BRUTES_PATH + "\\UAVIDS-2025"
NETTOYEES_PATH = r"C:\Drone_Attack_Similarity_Project\DATASET\Nettoyées"
TABLES_PATH    = r"C:\Drone_Attack_Similarity_Project\Rapport\tables"
FIGURES_PATH   = r"C:\Drone_Attack_Similarity_Project\Rapport\figures"
os.makedirs(NETTOYEES_PATH, exist_ok=True)
os.makedirs(TABLES_PATH, exist_ok=True)
os.makedirs(FIGURES_PATH, exist_ok=True)

# ─── Chargement UAVIDS-2025 ─────────────────────────────────────────────────
df_uavids = pd.read_csv(UAVIDS_PATH + "\\UAVIDS-2025.csv")

# ─── Chargement et fusion IoT/UAV Composite ─────────────────────────────────
FICHIERS_ATTAQUES = {
    "Bruteforce"    : ATTACKS_PATH + "\\BruteforceC.csv",
    "DDoS"          : ATTACKS_PATH + "\\DDoSC.csv",
    "DoS"           : ATTACKS_PATH + "\\DoSC.csv",
    "Evil Twin"     : ATTACKS_PATH + "\\EvilC.csv",
    "Fake Landing"  : ATTACKS_PATH + "\\FakelandingC.csv",
    "MITM"          : ATTACKS_PATH + "\\MITMC.csv",
    "Reconnassiance": ATTACKS_PATH + "\\ReconnassianceC.csv",
    "Reply"         : ATTACKS_PATH + "\\ReplyC.csv",
    "Scanning"      : ATTACKS_PATH + "\\ScanningC.csv",
    "Normal"        : NORMAL_PATH  + "\\Normal.csv",
}

fragments = []
for label, path in FICHIERS_ATTAQUES.items():
    df_tmp = pd.read_csv(path)
    if "label" not in df_tmp.columns:
        df_tmp["label"] = label
    fragments.append(df_tmp)

df_composite = pd.concat(fragments, ignore_index=True)

datasets_bruts = {
    "UAVIDS-2025"       : df_uavids,
    "IoT/UAV Composite" : df_composite,
}
print("Datasets bruts chargés.")


# ═══════════════════════════════════════════════════════════════════════════
# Fonction de nettoyage
# ═══════════════════════════════════════════════════════════════════════════
COLS_TO_DROP = ["FlowID", "SrcAddr", "DstAddr"]  # identifiants non pertinents

def nettoyer(df, name):
    avant = df.shape[0]
    # Suppression colonnes non pertinentes
    drop = [c for c in COLS_TO_DROP if c in df.columns]
    if drop:
        df = df.drop(columns=drop)
    # NaN
    df = df.dropna()
    # Doublons
    df = df.drop_duplicates()
    # Infinis
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    apres = df.shape[0]
    print(f"  {name:<22} : {avant:>8} → {apres:>8} lignes  (-{avant - apres:>7})")
    return df

print(f"{'Dataset':<22}   {'Avant':>8}    {'Après':>8}   {'Supprimées':>11}")
print("-" * 60)
datasets_clean = {}
for name, df in datasets_bruts.items():
    datasets_clean[name] = nettoyer(df.copy(), name)


# ═══════════════════════════════════════════════════════════════════════════
# Résumé du nettoyage
# ═══════════════════════════════════════════════════════════════════════════
resume = pd.DataFrame([
    {
        "Dataset"          : name,
        "Lignes_avant"     : datasets_bruts[name].shape[0],
        "Lignes_apres"     : datasets_clean[name].shape[0],
        "Lignes_supprimees": datasets_bruts[name].shape[0] - datasets_clean[name].shape[0],
        "Colonnes_apres"   : datasets_clean[name].shape[1],
    }
    for name in datasets_bruts
])
resume.to_csv(TABLES_PATH + "\\resume_nettoyage.csv", index=False)
print("Exporté : resume_nettoyage.csv")
resume

# ═══════════════════════════════════════════════════════════════════════════
# Sauvegarde des datasets nettoyés
# ═══════════════════════════════════════════════════════════════════════════
for name, df in datasets_clean.items():
    fname = name.replace("/", "_").replace(" ", "_") + "_clean.csv"
    df.to_csv(NETTOYEES_PATH + "\\" + fname, index=False)
    print(f"  Sauvegardé : {fname}  ({df.shape[0]} lignes × {df.shape[1]} colonnes)")
print(f"\n{len(datasets_clean)} fichiers sauvegardés dans DATASET/Nettoyées/")


# ═══════════════════════════════════════════════════════════════════════════
# Matrices de corrélation (une par dataset)
# ═══════════════════════════════════════════════════════════════════════════
for name, df in datasets_clean.items():
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()[:15]
    if len(num_cols) < 2:
        continue
    fig, ax = plt.subplots(figsize=(11, 9))
    mask = np.triu(np.ones((len(num_cols), len(num_cols)), dtype=bool))
    sns.heatmap(
        df[num_cols].corr(), mask=mask, annot=False,
        cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.2, ax=ax
    )
    fname_safe = name.replace("/", "_").replace(" ", "_")
    ax.set_title(f"Matrice de corrélation — {name}", fontweight="bold")
    plt.tight_layout()
    plt.savefig(FIGURES_PATH + f"\\Matrice_de_correlation_{fname_safe}.png", dpi=120)
    plt.close()
    print(f"  Matrice_de_correlation_{fname_safe}.png")


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Drone_Attack_Similarity_Project\\DATASET\\Brutes\\Each_Attacks_CSV\\BruteforceC.csv'